### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/thesis/11_Offline_Joint_Reference.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Ffiles%2Fnotebooks%2Fthesis%2F11_Offline_Joint_Reference.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Ffiles%2Fnotebooks%2Fthesis%2F11_Offline_Joint_Reference.ipynb)

- **Google Colab:** open the notebook, select a GPU for training under **Runtime > Change runtime type**, then choose **Run all**.
- **Kaggle:** sign in and import the notebook, enable **Internet**, select a **GPU** accelerator for training, then **Run all**.
- **Binder:** opens a temporary CPU JupyterLab session. Use it to inspect the notebook or run small checks; full training needs more resources.
- **[Studio Lab](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/thesis/11_Offline_Joint_Reference.ipynb) (existing accounts only):** start a runtime, copy the notebook to your project, select a Python **3.11–3.13** kernel, and set `RUNTIME = "studiolab"` in the first code cell before **Run all**. For CPU, also set `CUDA = False`.

The **first code cell** finds or downloads the repository and prepares TensorFlow **2.20** / Keras **3.11.2** before project imports. If setup requests a restart, restart the kernel and run all again. For another hosted Jupyter service, set `RUNTIME = "hosted"` (`CUDA = False` for CPU or compatible provider-managed CUDA). Locally, select the project TensorFlow kernel.

Launch links open the published GitHub `main` version; publish this notebook and its setup files together before using them. For a notebook that has not been published, upload its `.ipynb` file to Colab or Kaggle instead. GPU availability depends on the provider. Save checkpoints and results before a temporary session ends.

Benchmark training and collection notebooks require the campaign artifacts prepared in notebook **01**. This supplemental reference runs independently.

See the [hosted runtime guide](https://github.com/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/files/notebooks/thesis/README.md#hosted-runtimes) for setup and import details.


In [1]:
"""Prepare a notebook checkout and runtime before importing scientific packages.

This canonical first-cell source is copied into maintained notebooks and generated
HPO notebooks. Edit REVISION/RUNTIME/CUDA before execution when needed. Existing
checkouts are reused without fetching; only missing setup code is downloaded.
ROOT is the selected absolute pathlib.Path; RUNTIME_PACKAGES maps package names
to installed version strings. Preparation may clone/install under the selected
policy and changes cwd/import paths through files.notebooks.init.prepare_notebook.
"""
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "auto"  # Use "hosted" for another online service, or "local" to verify only.
CUDA = None  # False: CPU or managed CUDA; True: retain CUDA pip dependencies.

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME, 
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "files" / "notebooks" / "init.py" for path in _locations
                     if (path / "files" / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/files/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, 
    revision=REVISION, 
    runtime=RUNTIME, 
    cuda=CUDA
)


Repository ready: /workspace/Continual-Learning-with-Diffusion-Vision-Transformers (runtime: local)
Runtime packages ready: Python 3.12.3, TensorFlow 2.20.0, Keras 3.11.2


# 11 - Offline joint-training reference

Train one fresh model on the pooled training data from every task, then evaluate it on the same task groups as the sequential experiments. This measures an offline reference for the cost of sequential data access.

Select **TensorFlow 2.20 (Docker GPU)**, restart the kernel, then run top to bottom. These notebooks create supplemental reference results; no measured results are included yet.

## 1. Check the runtime

The maintained environment is TensorFlow 2.20 with Keras 3. Run this notebook after other GPU training has finished.

In [2]:
import os


os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

from IPython.display import display
from files.notebooks.thesis.workflow import check_runtime


print(check_runtime())
from files.notebooks.thesis.reference_benchmarks import (
    configure_reference, prepare_reference, train_reference, finish_reference
)

2026-10-12 18:46:23.557001: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


{'TensorFlow': '2.20.0', 'Keras': '3.11.2', 'GPUs': 1}


## 2. Select a paired experiment

The defaults remain **CIFAR-10, seed 17, validation evaluation**. Choose CIFAR-100 in the same cell to run its reference. These defaults do not register a separate fixed reference campaign or automatically pair this run with the benchmark seeds. For a comparison, match the seed, class order, training partition, evaluation split and scoring policy.

Use `test` only after fixing the supplemental reference recipe and scope. Such a run remains outside the 24-stream benchmark and its collector. The known prior test exposure from historical HPO still applies; a fixed recipe and fresh seed do not create independent confirmation.

In [3]:
DATASET = "cifar10"  # "cifar10" or "cifar100"
SEED = 17  # Match the comparison run's seed and class order.
EVALUATION_SPLIT = "validation"  # Set "test" only after fixing the recipe.

In [4]:
BENCHMARK = 'offline_joint'
config = configure_reference(
    DATASET, BENCHMARK, evaluation_split=EVALUATION_SPLIT, seed=SEED
)

## 3. Review the training plan

All permitted training rows are available from the start and are mixed across tasks in one fit. The task groups are retained for evaluation; they are not training stages. No replay, stored-example buffer, knowledge distillation, or semantic route is used.

Both references retain the thesis DiT architecture and combined diffusion/classification objective. The current recipe uses Adam at initial LR **0.001**, zero weight decay, **50 epochs** and batch size **128**. Evaluation uses raw primary-head predictions only (`clf_acc_coef=1.0`, `clf_distil_acc_coef=0.0`), averaged uniformly across **64 retained timesteps** from `max_t=256`, `t_range_drop_rate=0.75`. Inverse-SNR weighting selects timesteps for removal; it does not weight the retained predictions (`weighted=false`).

During `prepare_reference`, the cosine duration is resolved from actual permitted training rows before optimizer creation, replacing the inherited replay-platform horizon. With the default uncapped 20% validation split, either dataset has 40,000 training rows: `50 × ceil(40000 / 128) = 15,650` updates, including the final partial batch each epoch. The prepared configuration and recorded `training_budget` are authoritative when splits or data limits change. Equal nominal epochs per permitted example do not establish equal compute with replay methods.

In [5]:
schedule = config.continually_learn
display({
    "dataset": config.dataset.name, 
    "reference": BENCHMARK, 
    "seed": schedule.seed, 
    "evaluation_split": EVALUATION_SPLIT, 
    "epochs_per_training_example": config.training.epochs, 
    "batch_size": config.dataset.batch_size, 
    "class_order": schedule.class_order, 
    "task_groups": schedule.task_groups
})

{'dataset': 'cifar10',
 'reference': 'offline_joint',
 'seed': 17,
 'evaluation_split': 'validation',
 'epochs_per_training_example': 50,
 'batch_size': 128,
 'class_order': [4, 0, 1, 7, 8, 6, 2, 9, 5, 3],
 'task_groups': [[4, 0], [1, 7], [8, 6], [2, 9], [5, 3]]}

## 4. Prepare data, model, and result folder

The helper applies the selected seed, class schedule and held-out split, then resolves the reference's actual cosine duration before creating its optimizer. Do not change the configuration after preparation.

In [6]:
context = prepare_reference(config)
print("Results folder:", context["run_dir"])

2026-10-12 18:46:32.743094: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1791830792.744365    2187 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79194 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:87:00.0, compute capability: 8.0


Model: "di_t_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ depth_0_patch_embedder          │ (None, 64, 128)        │        75,648 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depth_0_time_embedder           │ (None, 128)            │       128,000 │
│ (ConditionEmbedding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depth_0_label_embedder          │ (None, 128)            │         1,408 │
│ (ConditionEmbedding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depth_0_time_label_merger (Add) │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depth_1_encoder_block           │ (None, 64, 128)        │       231,046 │
│ (VisionTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depth_2_encoder_block           │ (None, 64, 128)        │       231,046 │
│ (VisionTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depth_3_encoder_block           │ (None, 64, 128)        │       231,046 │
│ (VisionTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depth_4_encoder_block           │ (None, 64, 128)        │       231,046 │
│ (VisionTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depth_5_encoder_block           │ (None, 64, 128)        │       231,046 │
│ (VisionTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depth_6_encoder_block           │ (None, 64, 128)        │       231,046 │
│ (VisionTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ unpatchifier (Functional)       │ (None, 32, 32, 3)      │        39,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ clf_depth_0_cls_token           │ (None, 1, 128)         │           256 │
│ (SingleTokenLayer)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ clf_depth_0_distil_token        │ (None, 1, 128)         │           256 │
│ (SingleTokenLayer)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ clf_depth_1_feature_aggregator  │ (None, 64, 128)        │             0 │
│ (FeatureHandler)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ clf_depth_1_encoder_block       │ (None, 66, 128)        │       231,046 │
│ (VisionTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ clf_depth_2_encoder_block       │ (None, 66, 128)        │       231,046 │
│ (VisionTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ clf_depth_3_encoder_block       │ (None, 66, 128)        │       231,046 │
│ (VisionTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 3,052,940 (11.65 MB)

 Trainable params: 2,924,868 (11.16 MB)

 Non-trainable params: 128,072 (500.56 KB)

Results folder: /workspace/Continual-Learning-with-Diffusion-Vision-Transformers/files/results/thesis_route_one/reference_benchmarks/cifar10/offline_joint/seed-17-20261012T184626Z-1112940b


## 5. Train once

This is the full reference run and may take substantial time. A repeated training-cell execution is rejected for the same prepared context. For a fresh experiment, restart the kernel and run from the beginning.

In [7]:
# The helper refuses a second training call for the same prepared context.
history = train_reference(config, context)

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py:857: UserWarning: Gradients do not exist for variables ['di_t_classifier/distil_classes/distil_classes__first_layer/kernel', 'di_t_classifier/distil_classes/distil_classes__first_layer/bias', 'di_t_classifier/distil_classes/distil_classes__final_layer/kernel', 'di_t_classifier/distil_classes/distil_classes__final_layer/bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(
2026-10-12 18:47:05.141947: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d4034034350 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-10-12 18:47:05.141983: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2026-10-12 18:47:06.115757: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set e

  3/313 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - classifier_accuracy: 0.0812 - classifier_loss: 2.3064 - loss: 3.3049 - noise_loss: 0.9985    

I0000 00:00:1791830889.760522    2659 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


310/313 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - classifier_accuracy: 0.1914 - classifier_loss: 2.1429 - loss: 2.4919 - noise_loss: 0.3489

2026-10-12 18:48:25.301106: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-10-12 18:48:25.301305: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-10-12 18:48:25.301335: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-10-12 18:48:25.301386: I external/l

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - classifier_accuracy: 0.1919 - classifier_loss: 2.1419 - loss: 2.4890 - noise_loss: 0.3471

2026-10-12 18:49:08.138476: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-10-12 18:49:08.138529: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-10-12 18:49:10.702126: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_54', 12 bytes spill stores, 12 bytes spill loads

2026-10-12 18:49:20.801066: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filt

313/313 ━━━━━━━━━━━━━━━━━━━━ 182s 291ms/step - classifier_accuracy: 0.2440 - classifier_loss: 2.0347 - loss: 2.1955 - noise_loss: 0.1608 - val_classifier_accuracy: 0.3651 - val_classifier_loss: 1.7537 - val_image_loss: 0.0122 - val_loss: 1.8810 - val_noise_loss: 0.1273 - learning_rate: 9.9901e-04
Epoch 2/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 110ms/step - classifier_accuracy: 0.3273 - classifier_loss: 1.8466 - loss: 1.9125 - noise_loss: 0.0659 - val_classifier_accuracy: 0.4083 - val_classifier_loss: 1.6193 - val_image_loss: 0.0130 - val_loss: 1.7492 - val_noise_loss: 0.1300 - learning_rate: 9.9606e-04
Epoch 3/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 37s 117ms/step - classifier_accuracy: 0.3508 - classifier_loss: 1.7848 - loss: 1.8460 - noise_loss: 0.0612 - val_classifier_accuracy: 0.4495 - val_classifier_loss: 1.5151 - val_image_loss: 0.0174 - val_loss: 1.6889 - val_noise_loss: 0.1738 - learning_rate: 9.9114e-04
Epoch 4/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 37s 119ms/step - classifier_accuracy: 0.3657 -

## 6. Save and inspect results

Inspect final overall accuracy and the per-task evaluation table. **Forgetting and backward transfer are not applicable**: this model has no sequence of task-end states. They must remain unavailable, not be reported as zero.

In [8]:
summary, per_task = finish_reference(config, context, history)
display(summary)
display(per_task)
print("Saved results:", context["run_dir"])

2026-10-12 19:24:35.800057: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-10-12 19:28:20.086456: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-10-12 19:35:55.219580: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


{'benchmark': 'offline_joint',
 'dataset': 'cifar10',
 'seed': 17,
 'evaluation_split': 'validation',
 'metric_scale': 'fraction',
 'final_average_accuracy': 0.6697000026702881,
 'average_incremental_accuracy': None,
 'average_forgetting': None,
 'backward_transfer': None,
 'final_example_accuracy': 0.6697000026702881,
 'accuracy_source': 'ensemble',
 'ensemble_accuracy_kwargs': {'network_name': 'raw',
  'max_t': 256,
  't_range_drop_rate': 0.75,
  'clf_distil_acc_coef': 0.0,
  'clf_acc_coef': 1.0},
 'training_seconds': 2055.8445114418864,
 'training_seconds_scope': 'Entire native train_model call, including any task evaluation, reporting and checkpoint writes; excludes preparation and final reference reporting.',
 'native_results_path': '/workspace/Continual-Learning-with-Diffusion-Vision-Transformers/files/results/thesis_route_one/reference_benchmarks/cifar10/offline_joint/seed-17-20261012T184626Z-1112940b/native/2026-10-12_18-46-38 offline_joint-cifar10-seed-17',
 'objective': 'Shar

,task,original_classes,accuracy,accuracy_percent
0,1,"[4, 0]",0.6645,66.450000
1,2,"[1, 7]",0.7490,74.900001
2,3,"[8, 6]",0.7725,77.249998
3,4,"[2, 9]",0.6370,63.700002
4,5,"[5, 3]",0.5255,52.550000


Saved results: /workspace/Continual-Learning-with-Diffusion-Vision-Transformers/files/results/thesis_route_one/reference_benchmarks/cifar10/offline_joint/seed-17-20261012T184626Z-1112940b


## Interpretation and cleanup

Treat this result as an empirical offline upper reference for the selected model and recipe. It is not the maximum possible accuracy, and another method can exceed it. The pooled model sees future tasks immediately, so it cannot establish the performance of an online learner.

After the final cell succeeds, **save this notebook with its outputs, then restart its kernel** to release its training state before running another notebook. Result artifacts are saved separately in the printed folder.

See the [reference definitions and comparison limits](README.md#offline-and-naive-reference-benchmarks).